# EEGNet on Colab GPU

Run the EEGNet batch (14 subjects × 5-fold CV × 40 epochs) on a Colab T4 GPU instead of CPU. ~10–20 min total.

## One-time setup before running this notebook

1. **Enable GPU**: `Runtime` → `Change runtime type` → `T4 GPU` (or any GPU) → Save.
2. **Upload the preprocessed .fif files to Google Drive**:
   - In Drive, create the folder path `MyDrive/eeg-speech-data/processed/`
   - Drag the 14 `*-clean-epo.fif` files (MM05, MM08-MM12, MM14-MM16, MM18-MM21, P02) into it
   - Total ~1.7 GB. Wait for the upload to fully sync before continuing.
3. Run all cells below.

In [ ]:
# 1. Confirm GPU is attached
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime → Change runtime type → GPU, then Run all again.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'torch {torch.__version__}, CUDA {torch.version.cuda}')

In [ ]:
# 2. Mount Drive (auth prompt) and verify the preprocessed .fif files are there
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/eeg-speech-data/processed'
OUT_DIR  = '/content/drive/MyDrive/eeg-speech-data/outputs'

import os
if not os.path.isdir(DATA_DIR):
    raise SystemExit(f'Drive folder missing: {DATA_DIR}\n'
                     f'Create it in Drive (MyDrive/eeg-speech-data/processed) and upload the 14 .fif files first.')
fifs = sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.fif'))
assert len(fifs) == 14, f'Expected 14 .fif files in {DATA_DIR}, found {len(fifs)}: {fifs}'
print(f'Found {len(fifs)} preprocessed .fif files in Drive ✅')

In [ ]:
# 3. Clone the repo (idempotent — safe to re-run; pulls latest if already cloned)
import os, subprocess, sys

REPO_DIR = '/content/eeg-speech'
REPO_URL = 'https://github.com/joshegreenfield2/eeg-speech.git'

def run(cmd, cwd=None):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.stderr: print(r.stderr, file=sys.stderr)
    if r.returncode != 0:
        raise SystemExit(f'Command failed (exit {r.returncode}): {cmd}')

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print(f'{REPO_DIR} already cloned, pulling latest...')
    run('git pull', cwd=REPO_DIR)
else:
    if os.path.exists(REPO_DIR):
        print(f'{REPO_DIR} exists but is not a git repo — removing')
        run(f'rm -rf {REPO_DIR}')
    run(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'\ncwd: {os.getcwd()}')
print(f'HEAD: {subprocess.check_output(["git","rev-parse","--short","HEAD"], text=True).strip()}')

# Sanity check: scripts/run_dl.py must exist
assert os.path.isfile('scripts/run_dl.py'), 'scripts/run_dl.py not found after clone!'
print('scripts/run_dl.py present ✅')

In [ ]:
# 4. Install EEG-specific deps not pre-installed on Colab.
#    (torch, numpy, scipy, sklearn, pandas, matplotlib, h5py, PyWavelets are already there.)
#    The pip 'dependency conflict' warnings about ipython/jedi or moviepy/decorator are harmless
#    — they're complaints about Colab's pre-installed packages, not failures.
!pip install -q mne mne-icalabel autoreject pyprep braindecode==0.8.1 mat73

In [ ]:
# 5. Symlink data/processed → Drive folder (matches the script's expected path)
import os, glob

assert os.getcwd().endswith('eeg-speech'), f'Wrong cwd: {os.getcwd()} — re-run cell 3'
os.makedirs('data', exist_ok=True)
os.makedirs('outputs/results', exist_ok=True)

link_path = 'data/processed'
if os.path.islink(link_path):
    os.remove(link_path)
elif os.path.exists(link_path):
    raise SystemExit(f'{link_path} exists and is not a symlink — please remove manually')
os.symlink(DATA_DIR, link_path)

found = glob.glob('data/processed/*.fif')
print(f'Symlink resolves to {len(found)} .fif files (expecting 14)')
assert len(found) == 14, 'Symlink/data setup wrong'

In [ ]:
# 6. Run EEGNet on all 14 subjects. Auto-detects CUDA via the device patch in src/train.py.
import os
assert os.path.isfile('scripts/run_dl.py'), 'run_dl.py missing — re-run cell 3'
!python -u scripts/run_dl.py --model eegnet --epochs-max 40

In [ ]:
# 7. Copy results back to Drive so they survive the Colab session
import shutil, glob, os
os.makedirs(OUT_DIR, exist_ok=True)

phase4_dirs = sorted(glob.glob('outputs/results/phase4_dl_*'))
if not phase4_dirs:
    raise SystemExit('No phase4 results found — check cell 6 output for training errors.')
latest = phase4_dirs[-1]
dest = os.path.join(OUT_DIR, 'results', os.path.basename(latest))
shutil.copytree(latest, dest, dirs_exist_ok=True)
print(f'Copied {latest} → {dest}')

if os.path.exists('outputs/summary_results.csv'):
    shutil.copy('outputs/summary_results.csv', os.path.join(OUT_DIR, 'summary_results.csv'))
    print(f'Copied outputs/summary_results.csv → {OUT_DIR}/summary_results.csv')

print('\nDone. Pull the phase4_dl_* folder + summary_results.csv back to your local repo and run notebooks/03_results.ipynb.')